# Aquaplanet

SPEEDY T31L8 over a slab ocean with thermodynamic sea ice. This run is one command:

```bash
python -m jem.main +configuration=aquaplanet-slab
```

The notebook below composes the same configuration in Python and calls `jem.runners.run(cfg)` -- the entry point `python -m jem.main` itself uses -- so it can plot what the run wrote.

In [ ]:
from pathlib import Path

from hydra import compose, initialize_config_module

import jem.config  # noqa: F401  -- registers the ${jem_data:}/${jcm_data:} resolvers
from jem import plot, runners

output_dir = (Path("output") / "01-01_aquaplanet").resolve()
output_dir.mkdir(parents=True, exist_ok=True)

## Run it

In [ ]:
with initialize_config_module(config_module="jem.config", version_base="1.3"):
    cfg = compose(config_name="config", overrides=[
        "+configuration=aquaplanet-slab",
        f"coupled_run.output_dir={output_dir}",
        "coupled_run.subsample=3",       # 10 records out of 30 coupled days
        "coupled_run.checkpoint_path=null",
    ])
result = runners.run(cfg)
result.steps_completed, [p.name for p in result.paths]

## What it wrote

One file per component per chunk, named after the coupled step its chunk starts at (`<component>-<first step>.nc`).

In [ ]:
atm = plot.open_output(output_dir, "atm")
ocn = plot.open_output(output_dir, "ocn")
seaice = plot.open_output(output_dir, "seaice")
list(ocn.data_vars)

## Plot

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(11, 4))

# `level` is a sigma coordinate, surface-first, so selecting by its
# value (rather than `.isel(level=0)`) says so without relying on
# that ordering (see CLAUDE.md's "Inspecting model output").
humidity = atm["specific_humidity"].sel(level=1.0, method="nearest").isel(time=-1)
plot.map_plot(humidity, ax=axes[0], title="Surface specific humidity [kg/kg]")

sst = ocn["sea_surface_temperature"].isel(time=-1) - 273.15
plot.map_plot(sst, ax=axes[1], title="Sea surface temperature [°C]")
plt.tight_layout()

fig, ax = plt.subplots()
plot.area_mean(ocn["sea_surface_temperature"]).plot(ax=ax)
ax.set_ylabel("Area-mean SST [K]")

# A short animation of the same field, for the record.
ani = plot.animate_map(ocn["sea_surface_temperature"] - 273.15, title="SST [°C]")
ani.save(output_dir / "sst.gif", writer="pillow")